In [4]:
from pathlib import Path

# TODO: change this to the folder that contains your 34,332 PNGs (the top folder above H1L1 etc.)
IMAGES_ROOT = Path(r"C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0")

# sanity checks
print("IMAGES_ROOT exists?", IMAGES_ROOT.exists())
pngs = list(IMAGES_ROOT.rglob("*.png"))
print("PNG count found:", len(pngs))

# show a few example paths so we can infer folder structure
for p in pngs[:5]:
    print(p)


IMAGES_ROOT exists? True
PNG count found: 8472
C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1\1080Lines\H1_09HE6k6EaS_spectrogram_0.5.png
C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1\1080Lines\H1_09HE6k6EaS_spectrogram_1.0.png
C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1\1080Lines\H1_09HE6k6EaS_spectrogram_2.0.png
C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1\1080Lines\H1_09HE6k6EaS_spectrogram_4.0.png
C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1\1080Lines\H1_0jdZllcAme_spectrogram_0.5.png


In [5]:
from pathlib import Path
IMAGES_ROOT = Path(r"C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0")
sorted([p.name for p in IMAGES_ROOT.iterdir() if p.is_dir()])[:20]


['H1L1']

In [6]:
from pathlib import Path
from collections import Counter

IMAGES_ROOT = Path(
    r"C:\Users\mygam\Documents\gravityspy-glitch-explorer\data\gravityspy_raw\trainingsetv1d0\H1L1"
)

pngs = list(IMAGES_ROOT.rglob("*.png"))

# true label = parent folder name (one level above the file)
labels = [p.parent.name for p in pngs]

print("Total PNGs:", len(pngs))
print("Number of unique glitch classes:", len(set(labels)))

print("\nTop 10 classes by image count:")
for label, count in Counter(labels).most_common(10):
    print(f"{label:25s} {count}")


Total PNGs: 8472
Number of unique glitch classes: 4

Top 10 classes by image count:
Blip                      6000
1080Lines                 1312
1400Ripples               928
Air_Compressor            232


In [7]:
import re
import pandas as pd
from pathlib import Path

# use what you already set
# IMAGES_ROOT = Path(r"...\H1L1")

ID_RE = re.compile(r"_[A-Za-z0-9]{10}_")  # matches _09HE6k6EaS_ pattern
IFO_RE = re.compile(r"^(H1|L1)_")

rows = []
for p in IMAGES_ROOT.rglob("*.png"):
    name = p.name
    # true label from folder
    true_label = p.parent.name

    # IFO from filename start (H1_ or L1_)
    m_ifo = IFO_RE.search(name)
    ifo = m_ifo.group(1) if m_ifo else None

    # gravityspy_id from filename middle: H1_<ID>_spectrogram_...
    # we'll grab the 10-char token between underscores
    parts = name.split("_")
    gid = parts[1] if len(parts) >= 3 else None

    rows.append({
        "image_path": str(p),
        "true_label": true_label,
        "ifo": ifo,
        "gravityspy_id": gid,
        "scale": parts[-1].replace(".png", "") if name.endswith(".png") else None
    })

df = pd.DataFrame(rows)

print("Rows:", len(df))
print("Null gravityspy_id:", df["gravityspy_id"].isna().sum())
print("Null ifo:", df["ifo"].isna().sum())
print("Unique gravityspy_id:", df["gravityspy_id"].nunique())

df.head(3)


Rows: 8472
Null gravityspy_id: 0
Null ifo: 0
Unique gravityspy_id: 2118


,image_path,true_label,ifo,gravityspy_id,scale
0,C:\Users\mygam\Documents\gravityspy-glitch-exp...,1080Lines,H1,09HE6k6EaS,0.5
1,C:\Users\mygam\Documents\gravityspy-glitch-exp...,1080Lines,H1,09HE6k6EaS,1.0
2,C:\Users\mygam\Documents\gravityspy-glitch-exp...,1080Lines,H1,09HE6k6EaS,2.0


In [8]:
# How many images per gravityspy_id?
counts = df.groupby("gravityspy_id").size()

print("Images per gravityspy_id (value counts):")
print(counts.value_counts().sort_index())

# Show any IDs that don't have exactly 4 images (if any)
weird = counts[counts != 4]
print("\nIDs with non-4 image counts:", len(weird))
if len(weird) > 0:
    print(weird.head(20))


Images per gravityspy_id (value counts):
4    2118
Name: count, dtype: int64

IDs with non-4 image counts: 0


In [9]:
# Collapse to one row per gravityspy_id
event_df = (
    df
    .groupby("gravityspy_id")
    .agg(
        true_label=("true_label", "first"),
        ifo=("ifo", "first"),
        image_paths=("image_path", list),
        scales=("scale", list),
    )
    .reset_index()
)

print("Event rows:", len(event_df))
print("Unique labels:", event_df["true_label"].value_counts())

event_df.head(3)


Event rows: 2118
Unique labels: true_label
Blip              1500
1080Lines          328
1400Ripples        232
Air_Compressor      58
Name: count, dtype: int64


,gravityspy_id,true_label,ifo,image_paths,scales
0,017BiNepgE,Blip,H1,[C:\Users\mygam\Documents\gravityspy-glitch-ex...,"[0.5, 1.0, 2.0, 4.0]"
1,02bQL8a3oy,Blip,H1,[C:\Users\mygam\Documents\gravityspy-glitch-ex...,"[0.5, 1.0, 2.0, 4.0]"
2,02r7PgJFWL,Blip,H1,[C:\Users\mygam\Documents\gravityspy-glitch-ex...,"[0.5, 1.0, 2.0, 4.0]"


In [10]:
from pathlib import Path

OUTDIR = Path("outputs")
OUTDIR.mkdir(exist_ok=True)

OUT_CSV = OUTDIR / "gravityspy_events_with_true_labels.csv"
event_df.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_CSV)
print("File exists?", OUT_CSV.exists())


Wrote: outputs\gravityspy_events_with_true_labels.csv
File exists? True


In [11]:
import pandas as pd

check_df = pd.read_csv("outputs/gravityspy_events_with_true_labels.csv")

print("Rows:", len(check_df))
print("\nLabel counts:")
print(check_df["true_label"].value_counts())


Rows: 2118

Label counts:
true_label
Blip              1500
1080Lines          328
1400Ripples        232
Air_Compressor      58
Name: count, dtype: int64


In [12]:
# Focus only on clusters where the top label is Blip
blip_clusters = (
    cluster_stats
    .reset_index()
    .query("top_label == 'Blip'")
    .sort_values("n", ascending=False)
)

blip_clusters[["cluster_id", "n", "purity"]].head(10)


NameError: name 'cluster_stats' is not defined